In [ ]:
# Cell 1: Setup
import sys, os, time, math, random, subprocess
from pathlib import Path

def add_project_root(marker_dir="src", max_hops=5):
    p = Path.cwd().resolve()
    for _ in range(max_hops):
        if (p / marker_dir).exists():
            sys.path.insert(0, str(p))
            print(f"[OK] Added to sys.path: {p}")
            return
        p = p.parent
    raise RuntimeError(f"Could not find '{marker_dir}' within {max_hops} parents from {Path.cwd()}")

add_project_root("src")

import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd

# Project imports & model builders
from src.data.prepare_dataset import build_pairs_for_split
from src.data.dataloader import make_loaders
from src.data.augmentations import get_val_augs

# Baseline: adjust import to your baseline class if different
from src.models.unet import UNet as BaselineUNet

# Proposed wrapper
from src.models.wrappers.dpcn_concat_unet import DPCNConcatUNet

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 1337
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print("Device:", DEVICE)

[OK] Added to sys.path: D:\Data\Projects\Thesis\Adaptive-Hierarchical-Feature-Modulation-U-Net-Model-for-Retinal-Vessel-Segmentation
Device: cuda


In [20]:
# Configuration 
 
# ---- datasets ----
DATASETS = {
    "DRIVE":     dict(root="../../data/raw/DRIVE",     label_folder="1st_manual"),
    "CHASE-DB1": dict(root="../../data/raw/CHASE_DB1", label_folder="1st_manual"),
    "STARE":     dict(root="../../data/raw/STARE",     label_folder="2nd_manual"),
}

# ---- checkpoints ----
CKPTS = {
    "DRIVE": {
        "unet":     "../../outputs/checkpoints/unet/[DRIVE_UNET] base_unet.pth",
        "proposed": "../../outputs/checkpoints/[DRIVE] MATHFI.pth",
    },
    "CHASE-DB1": {
        "unet":     "../../outputs/checkpoints/unet/[CHASEDB1_UNET] base_unet.pth",
        "proposed": "../../outputs/checkpoints/[CHASEDB1] MATHFI.pth",
    },
    "STARE": {
        "unet":     "../../outputs/checkpoints/unet/[STARE_UNET] base_unet.pth",
        "proposed": "../../outputs/checkpoints/[STARE] MATHFI.pth",
    },
}

# ---- evaluation hyperparams ----
IMAGE_SIZE = 512      # val/test preprocessing size
TAU        = 0.5      # decision threshold for binarizing probs
NUM_WORKERS = 0       # safe locally; bump on Linux if desired
BATCH_SIZE  = 1
WINDOW      = IMAGE_SIZE   # sliding-window tile size
OVERLAP     = 0.25

# ---- proposed model (DPCN) hyperparams per dataset (MUST MATCH TRAINING) ----
# If trained the same way for all datasets, keep a single block.
PROPOSED_CFG = {
    "default": dict(enh_channels=64, iters=6, threshold_mode="scaled_vat", half_life=2.0, reduce_to=64),
    # Example: override per dataset (uncomment and edit if needed)
    # "DRIVE":   dict(enh_channels=32, iters=8, threshold_mode="scaled_vat", half_life=4.2, reduce_to=56),
    # "STARE":   dict(enh_channels=64, iters=6, threshold_mode="scaled_vat", half_life=2.8, reduce_to=64),
}

# ---- base UNet extras (as used in your code) ----
BASE_KW = {"cbam_reduction": 16}

# Where to dump CSVs (optional)
OUT_DIR = Path("./eval_exports"); OUT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def build_model(model_name: str, dataset_name: str):
    if model_name == "unet":
        m = BaselineUNet(in_channels=1)
    elif model_name == "proposed":
        cfg = PROPOSED_CFG.get(dataset_name, PROPOSED_CFG["default"])
        m = DPCNConcatUNet(
            in_ch=1,
            enh_channels=cfg["enh_channels"],
            iters=cfg["iters"],
            threshold_mode=cfg["threshold_mode"],
            half_life=cfg["half_life"],
            reduce_to=cfg["reduce_to"],
            base_kwargs=BASE_KW
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return m.to(DEVICE).eval()

def load_weights(model: torch.nn.Module, path: str):
    if not Path(path).exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")
    state = torch.load(path, map_location=DEVICE)
    model.load_state_dict(state)
    return model.eval()


In [3]:
# Build a test loader for each dataset (full-image mode)

def make_test_loader_for_dataset(root, label_folder, image_size=512, batch_size=1, num_workers=0, seed=SEED):
    test_pairs = build_pairs_for_split(root, split="test", label_folder=label_folder)

    # Pass test_pairs as both train/val to get a pure test loader (deterministic)
    _, test_loader = make_loaders(
        train_pairs=test_pairs,
        val_pairs=test_pairs,
        image_size=image_size,
        batch_size=batch_size,
        num_workers=num_workers,
        seed=seed,
        strict_fov=True,
        augs_train=None,
        augs_val=get_val_augs(image_size),
        patch_train=False
    )
    return test_loader


In [4]:
# Sliding window inference (averages logits)

@torch.no_grad()
def sliding_window_forward_logits(model, img_1chw, window=512, overlap=0.25, device=DEVICE):
    model.eval()
    img = img_1chw.to(device, non_blocking=True)
    _, _, H, W = img.shape
    step = max(1, int(window * (1 - overlap)))

    acc  = torch.zeros_like(img, device=device)
    norm = torch.zeros_like(img, device=device)

    for y0 in range(0, max(1, H - window + 1), step):
        for x0 in range(0, max(1, W - window + 1), step):
            y1 = min(y0 + window, H); x1 = min(x0 + window, W)
            y0 = y1 - window;         x0 = x1 - window
            tile = img[:, :, y0:y1, x0:x1]
            with torch.amp.autocast(device_type="cuda", enabled=(device=="cuda")):
                logit = model(tile)
            acc[:, :, y0:y1, x0:x1] += logit
            norm[:, :, y0:y1, x0:x1] += 1.0

    return acc / torch.clamp_min(norm, 1.0)


In [5]:
# Metrics (SEN, SPE, clDice) computed inside FOV

def _mask_flatten(p, t, m=None):
    if m is None: return p.view(-1), t.view(-1)
    return (p*m).view(-1), (t*m).view(-1)

@torch.no_grad()
def sensitivity_specificity(pred01, targ01, mask01=None, eps=1e-6):
    p, t = _mask_flatten(pred01, targ01, mask01)
    tp = (p*t).sum()
    tn = ((1-p)*(1-t)).sum()
    fp = (p*(1-t)).sum()
    fn = ((1-p)*t).sum()
    sen = (tp + eps) / (tp + fn + eps)
    spe = (tn + eps) / (tn + fp + eps)
    return float(sen), float(spe)

def cldice_score(pred01, targ01, mask01=None, eps=1e-6):
    try:
        from skimage.morphology import skeletonize
    except Exception:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "scikit-image", "-q"])
        from skimage.morphology import skeletonize

    pr = pred01.detach().cpu().numpy()[0,0].astype(np.uint8)
    gt = targ01.detach().cpu().numpy()[0,0].astype(np.uint8)
    if mask01 is not None:
        mm = mask01.detach().cpu().numpy()[0,0].astype(np.uint8)
        pr = (pr*mm).astype(np.uint8); gt = (gt*mm).astype(np.uint8)
    sp = skeletonize(pr>0).astype(np.uint8)
    sg = skeletonize(gt>0).astype(np.uint8)
    tprec = (sp & gt).sum() / (sp.sum() + eps)
    trec  = (sg & pr).sum() / (sg.sum() + eps)
    return float((2*tprec*trec) / (tprec + trec + eps))


In [6]:
# Evaluate a model on a loader

@torch.no_grad()
def evaluate_model_on_loader(model, loader, tau=0.5, window=512, overlap=0.25, device=DEVICE):
    sens, spes, clds = [], [], []
    for batch in loader:
        x   = batch["image"].to(device, non_blocking=True)   # [B,1,H,W]
        y   = batch["mask"].to(device,  non_blocking=True)   # [B,1,H,W]
        fov = batch.get("fov", torch.ones_like(y)).to(device)

        for i in range(x.size(0)):
            logits = sliding_window_forward_logits(model, x[i:i+1], window=window, overlap=overlap, device=device)
            probs  = torch.sigmoid(logits)
            pred01 = (probs >= tau).float()
            m = (fov[i:i+1] > 0.5).float()

            sen, spe = sensitivity_specificity(pred01, y[i:i+1], m)
            cld = cldice_score(pred01, y[i:i+1], m)

            sens.append(sen); spes.append(spe); clds.append(cld)

    return dict(SEN=float(np.mean(sens)), SPE=float(np.mean(spes)), clDice=float(np.mean(clds)))


In [21]:
# Run evals for all datasets & both models; build DataFrame

rows = []
for dname, cfg in DATASETS.items():
    print(f"\n=== {dname} ===")
    test_loader = make_test_loader_for_dataset(
        root=cfg["root"], label_folder=cfg["label_folder"],
        image_size=IMAGE_SIZE, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, seed=SEED
    )

    # U-Net
    m_unet = load_weights(build_model("unet", dname), CKPTS[dname]["unet"])
    r_unet = evaluate_model_on_loader(m_unet, test_loader, tau=TAU, window=WINDOW, overlap=OVERLAP, device=DEVICE)
    rows.append(dict(Dataset=dname, Model="U-Net", **r_unet))
    print("U-Net     :", r_unet)

    # Proposed
    m_prop = load_weights(build_model("proposed", dname), CKPTS[dname]["proposed"])
    r_prop = evaluate_model_on_loader(m_prop, test_loader, tau=TAU, window=WINDOW, overlap=OVERLAP, device=DEVICE)
    rows.append(dict(Dataset=dname, Model="Proposed", **r_prop))
    print("Proposed  :", r_prop)

df = pd.DataFrame(rows)
display(df.round(4))



=== DRIVE ===


RuntimeError: Error(s) in loading state_dict for UNet:
	Missing key(s) in state_dict: "d1.up.weight", "d1.up.bias", "d2.up.weight", "d2.up.bias", "d3.up.weight", "d3.up.bias", "d4.up.weight", "d4.up.bias". 
	Unexpected key(s) in state_dict: "d1.up.1.weight", "d1.up.2.weight", "d1.up.2.bias", "d1.up.2.running_mean", "d1.up.2.running_var", "d1.up.2.num_batches_tracked", "d2.up.1.weight", "d2.up.2.weight", "d2.up.2.bias", "d2.up.2.running_mean", "d2.up.2.running_var", "d2.up.2.num_batches_tracked", "d3.up.1.weight", "d3.up.2.weight", "d3.up.2.bias", "d3.up.2.running_mean", "d3.up.2.running_var", "d3.up.2.num_batches_tracked", "d4.up.1.weight", "d4.up.2.weight", "d4.up.2.bias", "d4.up.2.running_mean", "d4.up.2.running_var", "d4.up.2.num_batches_tracked". 
	size mismatch for final.weight: copying a param with shape torch.Size([2, 64, 1, 1]) from checkpoint, the shape in current model is torch.Size([1, 64, 1, 1]).
	size mismatch for final.bias: copying a param with shape torch.Size([2]) from

In [ ]:
# Deltas table (Proposed - U-Net)

def add_deltas(df):
    out = []
    for d in df["Dataset"].unique():
        a = df[(df.Dataset==d) & (df.Model=="Proposed")].iloc[0]
        b = df[(df.Dataset==d) & (df.Model=="U-Net")].iloc[0]
        out.append(dict(
            Dataset=d,
            dSEN=a.SEN-b.SEN,
            dSPE=a.SPE-b.SPE,
            dclDice=a.clDice-b.clDice
        ))
    return pd.DataFrame(out)

df_delta = add_deltas(df)
display(df_delta.round(4))


In [ ]:
# Matplotlib charts

def plot_grouped_metric(df, metric_name):
    ds = sorted(df['Dataset'].unique())
    x = np.arange(len(ds))
    width = 0.38

    vals_unet = [df[(df.Dataset==d)&(df.Model=="U-Net")][metric_name].values[0] for d in ds]
    vals_prop = [df[(df.Dataset==d)&(df.Model=="Proposed")][metric_name].values[0] for d in ds]

    plt.figure(figsize=(7.5,3.8))
    plt.bar(x - width/2, vals_unet, width, label='U-Net')
    plt.bar(x + width/2, vals_prop, width, label='Proposed')
    plt.xticks(x, ds)
    plt.ylabel(metric_name)
    plt.title(f"{metric_name}: U-Net vs Proposed")
    plt.ylim(0, 1.0)
    plt.grid(True, axis='y', linestyle=':', linewidth=0.8)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_deltas(df_delta):
    ds = sorted(df_delta['Dataset'].unique())
    x = np.arange(len(ds))
    width = 0.25
    dSEN = [df_delta[df_delta.Dataset==d]['dSEN'].values[0] for d in ds]
    dSPE = [df_delta[df_delta.Dataset==d]['dSPE'].values[0] for d in ds]
    dcl  = [df_delta[df_delta.Dataset==d]['dclDice'].values[0] for d in ds]

    plt.figure(figsize=(8.0,3.8))
    plt.bar(x - width, dSEN, width, label='ΔSEN')
    plt.bar(x,          dSPE, width, label='ΔSPE')
    plt.bar(x + width,  dcl,  width, label='ΔclDice')
    plt.xticks(x, ds)
    plt.axhline(0, color='k', linewidth=1)
    plt.ylabel('Delta (Proposed - U-Net)')
    plt.title('Metric Improvements over U-Net')
    plt.grid(True, axis='y', linestyle=':', linewidth=0.8)
    plt.legend()
    plt.tight_layout()
    plt.show()

for metric in ["SEN", "SPE", "clDice"]:
    plot_grouped_metric(df, metric)
plot_deltas(df_delta)
